# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

Part 4 trained the foundation model. The recurring production job is to run it over every card and store a fresh embedding for downstream models to consume. That's a heterogeneous job — CPU to read the tokens, GPU for the forward pass — and it streams through a single Ray Data pipeline.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Turn the trained model into one vector per transaction

The foundation model pretrained on long transaction histories, but the fraud model wants a fixed vector per transaction. Following NVIDIA's blueprint (their NB04) we embed each transaction **on its own** — feed `<bos>` + its field tokens + `<eos>` through the decoder and read the **last real token's** hidden state (`HuggingFaceDecoderInference`, `last_token` pooling).

Two choices happen here, and both matter:
- we **balance-sample the training set** (every fraud + a ~10% mix of normals) rather than embedding all ~19.5M train transactions, and embed the **100K stratified val/test eval subsets** from Part 2;
- the single-transaction context is what makes the embedding *complementary* to the raw features downstream (a 300-deep running average would just re-encode what the raw fields already say).

Each split is written as `embed_<split>.npy` + `lbl_<split>.npy` + `raw_<split>.parquet` (the 13 native features), so Part 6 can fit its classifiers without touching the tokenizer.

## Stream CPU work into GPU forward passes

This stage has two different appetites. Tokenizing transactions is CPU work; the forward pass is GPU work; and the model is expensive to load, so it should load once per GPU and stay put. The pipeline below gives each part its own hardware:

- a **CPU task** runs the seeded, order-sensitive preamble once per split — the balanced train sample (every fraud + normals to 1M, NVIDIA's NB01 recipe) and the sort that fixes output row order — and writes `lbl_`/`raw_` immediately, since labels and features never needed a GPU;
- **CPU workers** stream batches of transactions into token rows (`encode_txn_batch`, the same identity-verified tokenizer path as Part 3);
- **GPU actors** (`GPUEmbedder`, one per GPU, model loaded once in the constructor) do nothing but forward passes.

The two pools scale independently: more CPU workers keep the GPUs from starving, more GPUs raise embedding throughput linearly — and neither holds hardware the other needs. Each row carries its output position, so the assembled matrices are in the same order as the single-GPU reference — we verified the outputs match it (labels and features byte-equal, embeddings to float-kernel precision, `scripts/verify_distributed_embed.py`).

In [ ]:
from src.nvembed import (prepare_embed_split, encode_txn_batch, GPUEmbedder,
                         assemble_embeddings)

eb = cfg["embed"]
use_gpu = eb["use_gpu"]                    # mini: CPU actors; full: one actor per A10G
if not os.path.exists(os.path.join(paths["embeddings"], "embed_test.npy")):
    for split, fname in {"train": None, "val": "val_eval.parquet",
                         "test": "test_eval.parquet"}.items():
        # seeded preamble on one CPU task: balanced sample + order-fixing sort;
        # writes lbl_<split>.npy + raw_<split>.parquet directly
        prep = ray.get(prepare_embed_split.remote(
            paths["nvsplit"], fname, paths["embeddings"], split,
            eb["balanced_train"], 42))

        # stream: CPU workers tokenize -> GPU actors forward-pass -> shards
        shards = os.path.join(paths["embeddings"], f"_emb_{split}")
        ray.data.read_parquet(prep["prep"]) \
            .map_batches(encode_txn_batch, batch_format="pandas") \
            .map_batches(GPUEmbedder,
                         fn_constructor_kwargs={"hf_dir": paths["hf"],
                                                "batch_size": eb["batch_size"]},
                         batch_format="numpy", batch_size=4096,
                         num_gpus=(1 if use_gpu else 0),
                         concurrency=eb["num_workers"]) \
            .write_parquet(shards)

        meta = assemble_embeddings(shards, os.path.join(paths["embeddings"],
                                                        f"embed_{split}.npy"),
                                   prep["prep"], embed_dim=cfg["model"]["d_model"])
        print(split, {**{k: prep[k] for k in ("rows", "fraud")}, **meta})
else:
    print(f"embeddings present at {paths['embeddings']} — skipping (delete the dir to re-embed)")

## Did the embeddings actually learn anything?

The classic self-supervised failure is silent **representation collapse**: every customer maps to nearly the same vector while the loss looks fine. A cheap guard is to sample the embeddings and check two numbers — mean pairwise cosine similarity (→1.0 means collapse) and mean feature variance (→0 means collapse). `embedding_health` reports both.

In [3]:
# Representation-collapse check on the test embeddings: mean pairwise cosine similarity
# (→1.0 = collapse) and mean feature variance (→0 = collapse).
emb = np.load(os.path.join(paths["embeddings"], "embed_test.npy"))
lbl = np.load(os.path.join(paths["embeddings"], "lbl_test.npy"))

rng = np.random.RandomState(0)
idx = rng.choice(len(emb), size=min(2000, len(emb)), replace=False)
s = emb[idx].astype(np.float32)
s /= (np.linalg.norm(s, axis=1, keepdims=True) + 1e-8)
cos = s @ s.T
mean_cos = float((cos.sum() - len(idx)) / (len(idx) * (len(idx) - 1)))

print(f"{len(emb):,} test embeddings · dim {emb.shape[1]}  ({int(lbl.sum()):,} fraud)")
print(f"mean pairwise cosine  {mean_cos:.3f}   (→1.0 = collapse)")
print(f"mean feature variance {float(emb.var(0).mean()):.4f}   (→0 = collapse)")
print(f"example embedding[:8] = {emb[0, :8].round(3).tolist()}  (label {int(lbl[0])})")

100,000 test embeddings · dim 512  (112 fraud)
mean pairwise cosine  0.910   (→1.0 = collapse)
mean feature variance 0.4004   (→0 = collapse)
example embedding[:8] = [-1.6579999923706055, -0.7429999709129333, 0.6850000023841858, -0.8610000014305115, 0.4740000069141388, -2.2170000076293945, -0.43700000643730164, -0.1899999976158142]  (label 0)


## Takeaways

**Ray**
- The embed stage is a **streaming CPU→GPU pipeline**: CPU workers tokenize, GPU actors (model loaded once each) do only forward passes, and the two pools autoscale independently — the literal "CPU-bound steps scale separately from GPU-bound steps" pattern this series is built around.
- Output order is carried explicitly (each row's position travels with it), and the results are verified against the single-GPU reference: labels and raw features byte-equal, embeddings equal to float-kernel precision, and the downstream raw baseline reproduces to every digit (`scripts/verify_distributed_embed.py`).
- The notebook and the headless job compose the same pieces (`prepare_embed_split → encode_txn_batch → GPUEmbedder → assemble_embeddings`), so they cannot drift.

**The embeddings**
- One vector per transaction, from the **last real token** of a single-transaction sequence — aligned with the per-transaction fraud label and complementary to the raw fields.
- Written per split as `embed_/lbl_/raw_` so Part 6 fits its classifiers with no re-tokenization.
- A quick collapse check (mean pairwise cosine, feature variance) catches the silent self-supervised failure mode before it reaches the downstream model.

---

## Next

**Part 6 — Downstream fraud: raw vs embedding vs fusion**: fit NVIDIA's XGBoost recipe on three feature sets — raw transaction fields, the foundation model's embedding, and the two fused — and measure the lift at natural fraud prevalence with PR-AUC.